In [0]:
storage_account = "your_storage_account_name"
storage_key = "your_storage_account_key"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    avg,
    lag,
    col,
    stddev,
    round
)

# ===============================
# Read Silver Layer
# ===============================

silver_df = spark.read.parquet(
    f"abfss://silver@{storage_account}.dfs.core.windows.net/AAPL_clean.parquet"
)
# Ensure data is ordered by Date
silver_df = silver_df.orderBy("Date")

print("Silver Rows:", silver_df.count())

silver_df.printSchema()

display(silver_df.limit(5))

# ===============================
# Window Definitions
# ===============================

window = Window.orderBy("Date")

window5 = window.rowsBetween(-4, 0)
window20 = window.rowsBetween(-19, 0)

# ===============================
# Feature 1 : Daily Return
# ===============================

gold_df = silver_df.withColumn(
    "Daily_Return",
    (
        col("Close") -
        lag("Close", 1).over(window)
    ) /
    lag("Close", 1).over(window)
)

# ===============================
# Feature 2 : 5-Day Moving Average
# ===============================

gold_df = gold_df.withColumn(
    "MA_5",
    avg("Close").over(window5)
)

# ===============================
# Feature 3 : 20-Day Moving Average
# ===============================

gold_df = gold_df.withColumn(
    "MA_20",
    avg("Close").over(window20)
)

# ===============================
# Feature 4 : 20-Day Volatility
# ===============================

gold_df = gold_df.withColumn(
    "Volatility",
    stddev("Daily_Return").over(window20)
)

# ===============================
# Round Numeric Columns
# ===============================

gold_df = (
    gold_df
    .withColumn("Daily_Return", round(col("Daily_Return"), 6))
    .withColumn("MA_5", round(col("MA_5"), 4))
    .withColumn("MA_20", round(col("MA_20"), 4))
    .withColumn("Volatility", round(col("Volatility"), 6))
)

# ===============================
# Preview Gold Layer
# ===============================

display(gold_df)

# ===============================
# Save Gold Layer
# ===============================

gold_df.write.mode("overwrite").parquet(
    f"abfss://gold@{storage_account}.dfs.core.windows.net/AAPL_features.parquet"
)

print("Gold Layer created successfully.")

Silver Rows: 1255
root
 |-- Close: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Open: double (nullable = true)
 |-- Volume: long (nullable = true)
 |-- Date: timestamp_ntz (nullable = true)



Close,High,Low,Open,Volume,Date
145.2283172607422,146.04710543551434,143.9708787526298,144.5264945756414,72434100,2021-07-26T00:00:00
143.06434631347656,145.4427434254304,141.87514775749966,145.35500444699645,104818600,2021-07-27T00:00:00
141.31956481933594,143.25932698356266,138.94116715524945,141.15385873599058,118931200,2021-07-28T00:00:00
141.96286010742188,142.8498878684949,140.92962551695572,141.03684881635502,56699500,2021-07-29T00:00:00
142.1772918701172,142.63542638578699,140.4714762953773,140.73466343435456,70440600,2021-07-30T00:00:00


/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Close,High,Low,Open,Volume,Date,Daily_Return,MA_5,MA_20,Volatility
145.2283172607422,146.04710543551434,143.9708787526298,144.5264945756414,72434100,2021-07-26T00:00:00,null,145.2283,145.2283,null
143.06434631347656,145.4427434254304,141.87514775749966,145.35500444699645,104818600,2021-07-27T00:00:00,-0.0149,144.1463,144.1463,null
141.31956481933594,143.25932698356266,138.94116715524945,141.15385873599058,118931200,2021-07-28T00:00:00,-0.012196,143.2041,143.2041,0.001913
141.96286010742188,142.8498878684949,140.92962551695572,141.03684881635502,56699500,2021-07-29T00:00:00,0.004552,142.8938,142.8938,0.010537
142.1772918701172,142.63542638578699,140.4714762953773,140.73466343435456,70440600,2021-07-30T00:00:00,0.00151,142.7505,142.7505,0.009715
141.84591674804688,143.2398049839172,141.58272954059322,142.66470486730785,62880000,2021-08-02T00:00:00,-0.002331,142.074,142.5997,0.008515
143.6394500732422,144.30227426975748,141.51448305654833,142.12858155366985,64786600,2021-08-03T00:00:00,0.012644,142.189,142.7482,0.010392
143.2397918701172,144.058579914417,142.58670986762564,143.5517196242508,56368300,2021-08-04T00:00:00,-0.002782,142.5731,142.8097,0.009494
143.34701538085938,144.107320690727,142.47948676454712,143.26903344221475,46397700,2021-08-05T00:00:00,7.49E-4,142.8499,142.8694,0.00884
142.66371154785156,143.61063897996743,142.16584847464708,142.86872275200778,54126800,2021-08-06T00:00:00,-0.004767,142.9472,142.8488,0.008337


Gold Layer created successfully.


In [0]:
gold_df.write.mode("overwrite").parquet(
    f"abfss://gold@{storage_account}.dfs.core.windows.net/AAPL_features.parquet"
)

/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
